# Piloto 2026 - prueba M0 de pérdida aromática en equilibrio

## tl;dr

**Veredicto ejecutado:** `INFORMATIVE_NEGATIVE`.

Esta prueba elimina el parámetro libre de transferencia para contrastar dos modelos de partición y, en paralelo, corrige la base NTU del modelo con `kLa` líquido. El caso diagnóstico 26157-MIX-03 queda registrado en la salida ejecutada: `{"equilibrium_unifac": 24.237741552403676, "equilibrium_morakul": 48.77593614414894, "liquid_kla_unifac": 4.69209447263327, "observed_captured_ug": 614.557866106016}`.

El criterio de éxito exige mejorar al menos 30 % el error holdout de condensado para octanoato e isoamilo sin degradar más de 10 % el ajuste de vino.

## Context & Methods

### Key Assumptions

- La química líquida inicializa cada trayectoria; la primera muestra no entra al RMSE.
- Los holdouts completos son 26158 y 26211.
- El `rCO2` emitido proviene del modelo de CO2 validado previamente.
- `K=Cgas/Clíquido`; el modelo de equilibrio usa `lambda=K*(Qgas/VL)`.
- Morakul/Mouret representa composición mediante etanol y temperatura. Los coeficientes de acetatos provienen de mosto natural y los de octanoato de mosto sintético piloto.
- El tren de captura conserva las eficiencias históricas; por ello esta prueba aísla partición/transferencia, no valida todavía el condensador.
- Los puntos censurados se tratan como límites, no como ceros.

Fuentes primarias: Morakul et al. 2011, DOI 10.1016/j.procbio.2011.01.034; Mouret et al. 2014, DOI 10.1016/j.foodres.2014.02.044.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "fermentation_model").exists():
    ROOT = ROOT.parent
if not (ROOT / "fermentation_model").exists():
    raise RuntimeError("Execute from the repository or a descendant directory")
sys.path.insert(0, str(ROOT / "fermentation_model"))
from pilot_2026 import run_aroma_equilibrium_partition_test_2026 as analysis
result = analysis.load_results() if os.environ.get("PILOT_AROMA_REUSE_RESULTS") == "1" else analysis.run_analysis()
print("Resultados:", analysis.RESULTS_DIR.relative_to(ROOT))
print("Veredicto:", result["gate"]["verdict"])

## Data

In [ ]:
display(result["coefficients"])
display(result["pulses"][["batch", "pulse_time_h", "timing_source"]].round(3))
display(Image(filename=analysis.FIGURE_DIR / "partition_trajectories.png"))

## Results

In [ ]:
holdout = result["metrics"].query("fit_scope == 'calibration_only' and role == 'holdout'")
display(holdout[["model_variant", "species", "domain", "n_observed_scored", "rmse", "nrmse_over_mean", "bias"]].round(4))
display(result["comparison"].round(4))
display(Image(filename=analysis.FIGURE_DIR / "holdout_metric_comparison.png"))

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / "holdout_wine_predictions.png"))
display(Image(filename=analysis.FIGURE_DIR / "holdout_condensate_predictions.png"))

## Transfer and identifiability checks

In [ ]:
display(result["validation"].round(5))
display(result["parameters"].query("fit_scope == 'all_data'").round(6))
display(Image(filename=analysis.FIGURE_DIR / "ethyl_octanoate_effective_loss.png"))

## Takeaways

- Si equilibrio-Morakul mejora condensado y respeta el guardrail de vino, el siguiente paso es el estado de headspace/trampa.
- Si el modelo sigue fallando en MIX-04 o sobrepasa el techo calculado con química interpolada, el bloqueo está en producción temporal, captura o resolución de muestreo, no en un `kLa` adicional.
- Un `kLa` corregido que caiga al límite inferior indica que el optimizador intenta reconstruir artificialmente la penalización histórica.
- Ningún resultado de acetato de etilo permite declarar identificabilidad de pérdida porque todos sus condensados están censurados.

In [ ]:
assert result["gate"]["verdict"] in {"PASS", "INFORMATIVE_NEGATIVE", "COMPUTATIONAL_FAILURE"}
assert len(result["figures"]) == 5
print("Notebook ejecutado sin errores.")
print("Figuras embebidas:", len(result["figures"]))
print("Gate:", result["gate"]["verdict"])